# [Member 2] Transportation: SIRTA crashes — sampling, EDA and cleaning

**Dataset:** SIRTA Data Crash (`xeqp-qz8h`) — https://data.cityofnewyork.us/d/xeqp-qz8h
**Output:** `data/clean/transport_clean.csv` + `data/clean/transport_notes.md`
Run top to bottom. Everything printed here feeds the notebook sections *Transport EDA and cleaning* and the validation plots (Member 6).

## 1. Sampling: check the size first, then download a filtered sample

**Why a filter?** The full crash table is large and can crash a notebook. We keep crashes from **2022-01-01 onward** because
(1) Member 1's housing data has most of its completions/starts in 2022 and later, so both domains cover the same years;
(2) recent years reflect the current street network and current ZIP/borough reporting; (3) it is still hundreds of thousands of rows
across all five boroughs, so it represents the whole city. `$order=:id` makes the sample repeatable.

In [ ]:
import os, re, json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from urllib.parse import urlencode

TRANSPORT_ID   = 'xeqp-qz8h'
SINCE          = '2022-01-01T00:00:00'
RAW_PATH       = 'data/raw/transport_raw_2022_onward.csv'
CLEAN_PATH     = 'data/clean/transport_clean.csv'
NOTES_PATH     = 'data/clean/transport_notes.md'
for p in ('data/raw', 'data/clean'):
    os.makedirs(p, exist_ok=True)

DATE_COL_GUESS = 'crash_date'          # confirmed in Member 1's checks (crash_date, zipcode, crash_borough)
RESOURCE_URL   = 'https://data.cityofnewyork.us/resource/{}.csv'

def count_rows(dataset_id, where=None):
    """Ask the API how many rows match BEFORE downloading (Listing 2 safety check)."""
    params = {'$select': 'count(*)'}
    if where:
        params['$where'] = where
    n = pd.read_csv(RESOURCE_URL.format(dataset_id) + '?' + urlencode(params)).iloc[0, 0]
    return int(n)

def download_sample(dataset_id, where, limit=1_000_000):
    """Filtered download (Listing 2): $where filter, $limit above the row count, $order for a stable sample."""
    params = {'$limit': limit, '$where': where, '$order': ':id'}
    return pd.read_csv(RESOURCE_URL.format(dataset_id) + '?' + urlencode(params), low_memory=False)

where_clause = f"{DATE_COL_GUESS} >= '{SINCE}'"
if os.path.exists(RAW_PATH):
    df_raw = pd.read_csv(RAW_PATH, low_memory=False)
    print('Loaded saved raw sample:', df_raw.shape)
else:
    try:
        print('Rows in full dataset  :', count_rows(TRANSPORT_ID))
        print('Rows matching filter  :', count_rows(TRANSPORT_ID, where_clause))
    except Exception as e:
        print('Count check failed (not fatal):', e)
    df_raw = download_sample(TRANSPORT_ID, where_clause)
    df_raw.to_csv(RAW_PATH, index=False)       # raw copy is never edited afterwards
    print('Downloaded and saved raw sample:', df_raw.shape)

# Notes collected for transport_notes.md (every filter / drop / assumption goes through log())
NOTES = []
def log(msg):
    print('•', msg); NOTES.append(msg)
log(f"Sample: {TRANSPORT_ID} filtered with {where_clause}; raw rows downloaded = {len(df_raw):,}.")

Rows in full dataset  : 13754
Rows matching filter  : 13310
Downloaded and saved raw sample: (13310, 23)
• Sample: xeqp-qz8h filtered with crash_date >= '2022-01-01T00:00:00'; raw rows downloaded = 13,310.


## 2. Overview: rows, columns, types, date range, geography, known issues
The API returns lowercase_with_underscore names, which can differ from the website CSV, so we print the real columns first.

In [ ]:
t = df_raw.copy()
t.columns = t.columns.str.strip().str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
socrata_cols = [c for c in t.columns if c.startswith('_') or c.startswith(':')]
t = t.drop(columns=socrata_cols)

def find_col(candidates, frame=t):
    """Return the first candidate column that exists (the API names can differ from the CSV)."""
    return next((c for c in candidates if c in frame.columns), None)

COL = {
    'date'   : find_col(['crash_date', 'date', 'accident_date']),
    'borough': find_col(['crash_borough', 'borough', 'boro']),
    'zip'    : find_col(['zipcode', 'zip_code', 'zip']),
    'lat'    : find_col(['latitude', 'lat']),
    'lon'    : find_col(['longitude', 'lon', 'long']),
    'id'     : find_col(['collision_id', 'crash_id', 'unique_key', 'id']),
}
print('Resolved key columns:', COL)
assert COL['date'] and COL['borough'] and COL['zip'], 'Check df.columns and edit COL above'

overview = pd.DataFrame({'dtype': t.dtypes.astype(str), 'non_null': t.notna().sum(),
                         'null_pct': (t.isna().mean() * 100).round(2), 'n_unique': t.nunique()})
rows_raw = len(t)
print(f'Rows: {rows_raw:,} | Columns: {t.shape[1]} | Memory: {t.memory_usage(deep=True).sum()/1e6:.1f} MB')
dates_raw = pd.to_datetime(t[COL['date']], errors='coerce')
print('Date range:', dates_raw.min(), '->', dates_raw.max())
overview

Resolved key columns: {'date': 'crash_date', 'borough': 'crash_borough', 'zip': 'zipcode', 'lat': 'latitude', 'lon': 'longitude', 'id': None}
Rows: 13,310 | Columns: 23 | Memory: 6.9 MB
Date range: 2022-01-01 00:00:00 -> 2026-06-30 00:00:00


,dtype,non_null,null_pct,n_unique
open_data_crash_id,int64,13310,0.00,13309
crash_date,object,13310,0.00,1641
crash_time,object,13310,0.00,1728
crash_police_precinct,int64,13310,0.00,78
crash_borough,object,13255,0.41,5
total_non_severe_injuries,int64,13310,0.00,16
total_severe_injuries,int64,13310,0.00,8
total_fatalities,int64,13310,0.00,6
latitude,float64,10868,18.35,8570
longitude,float64,10868,18.35,8219


## 3. EDA (before cleaning)
Missing values, numeric distributions, category counts and crashes over time and space.

In [ ]:
null_before = (t.isna().mean() * 100).round(2).sort_values(ascending=False)
nb_plot = null_before[null_before > 0].rename_axis('column').reset_index(name='pct_missing')
if len(nb_plot):
    px.bar(nb_plot, x='column', y='pct_missing', title='Missing values per column BEFORE cleaning (%)').show()
else:
    print('No missing values')

# Numeric columns (injury / death / vehicle counts ...) -> descriptive statistics
numeric_cols = [c for c in t.select_dtypes('number').columns if c not in (COL['lat'], COL['lon'], COL['id'])]
numeric_cols = [c for c in numeric_cols if not re.search(r'zip|id$|code', c)]
print('Numeric columns:', numeric_cols)
display(t[numeric_cols].describe().T) if numeric_cols else None

Numeric columns: ['crash_police_precinct', 'total_non_severe_injuries', 'total_severe_injuries', 'total_fatalities', 'similar_fatalities_severe', 'similar_fatalities_all', 'community_board', 'council_district', 'bin', 'bbl', 'census_tract_2020']


,count,mean,std,min,25%,50%,75%,max
crash_police_precinct,13310.0,6.783787e+01,3.468395e+01,1.000000e+00,4.200000e+01,6.800000e+01,1.040000e+02,1.230000e+02
total_non_severe_injuries,13310.0,2.704733e-01,8.360310e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.900000e+01
total_severe_injuries,13310.0,1.011495e+00,4.515143e-01,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,7.000000e+00
total_fatalities,13310.0,8.279489e-02,2.943044e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00
similar_fatalities_severe,11917.0,9.549383e-02,4.217294e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.000000e+00
similar_fatalities_all,13310.0,1.544328e+00,5.857394e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.960000e+02
community_board,3427.0,2.746224e+02,1.173948e+02,1.010000e+02,2.020000e+02,3.040000e+02,4.010000e+02,5.030000e+02
council_district,3427.0,2.427429e+01,1.512823e+01,1.000000e+00,1.100000e+01,2.300000e+01,3.800000e+01,5.100000e+01
bin,2354.0,2.742248e+06,1.212782e+06,1.000000e+06,2.008285e+06,3.048364e+06,3.397377e+06,5.174401e+06
bbl,2354.0,2.653437e+09,1.166000e+09,1.000020e+09,2.026100e+09,3.014679e+09,3.079488e+09,5.200430e+09


In [ ]:
# Category columns with a manageable number of labels (borough, contributing factor, vehicle type ...)
cat_cols = [c for c in t.select_dtypes('object').columns if 1 < t[c].nunique() <= 60 and c != COL['date']]
print('Categorical columns:', cat_cols)
for c in cat_cols[:6]:
    print(f'\n--- {c} ---')
    print(t[c].value_counts(dropna=False).head(12))

Categorical columns: ['crash_borough']

--- crash_borough ---
crash_borough
Brooklyn         3954
Queens           3282
Manhattan        2733
Bronx            2617
Staten Island     669
NaN                55
Name: count, dtype: int64


In [ ]:
# Crashes by year / month / weekday / hour (date parsed on a temporary copy; the real parsing is in section 4)
eda = pd.DataFrame({'dt': dates_raw}).dropna()
eda['year'], eda['month'], eda['weekday'] = eda.dt.dt.year, eda.dt.dt.month, eda.dt.dt.day_name()
px.bar(eda['year'].value_counts().sort_index().reset_index(), x='year', y='count', title='Crashes per year (raw)').show()
px.bar(eda['month'].value_counts().sort_index().reset_index(), x='month', y='count', title='Crashes per calendar month (raw)').show()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
px.bar(eda['weekday'].value_counts().reindex(order).reset_index(), x='weekday', y='count', title='Crashes per weekday (raw)').show()
if (eda.dt.dt.hour > 0).any():   # only meaningful if the timestamp carries a time of day
    px.bar(eda.dt.dt.hour.value_counts().sort_index().reset_index(), x='dt', y='count', title='Crashes per hour (raw)').show()
else:
    print('crash_date has no time of day -> hourly analysis skipped (date only).')

# Initial spatial look: raw coordinates (includes bad points, which we handle in section 4)
if COL['lat'] and COL['lon']:
    pts = t[[COL['lat'], COL['lon']]].dropna()
    sample = pts.sample(min(20000, len(pts)), random_state=42)
    px.scatter(sample, x=COL['lon'], y=COL['lat'], opacity=0.3, title='Raw crash coordinates (20k sample)').show()

crash_date has no time of day -> hourly analysis skipped (date only).


## 4. Cleaning (each step logs what it changed, and why)
Order: blank rows → dates → borough → ZIP → coordinates → missing location handling → outliers → inconsistent categories → duplicates.

### 4.1 Blank rows and dates
Member 1's check showed the download contains rows that are entirely empty apart from system fields. A crash without a date cannot be placed in a
ZIP × year cell, so those rows are dropped. Dates are converted to datetime and we add `year`, `month`, `weekday` (and `hour` when the timestamp has a time).
Dates in the future are impossible for a crash that already happened, so they are removed.

In [ ]:
def clean_dates(frame, date_col):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors='coerce')
    return out

n0 = len(t)
t = clean_dates(t, COL['date'])
t = t[t[COL['date']].notna()].copy()
log(f"Dropped {n0 - len(t):,} rows with no valid crash date (blank rows / unparseable dates).")

today = pd.Timestamp.today().normalize()
future = t[COL['date']] > today
log(f"Dropped {future.sum():,} rows with a crash date in the future.")
t = t[~future].copy()

t['year']    = t[COL['date']].dt.year.astype(int)
t['month']   = t[COL['date']].dt.month.astype(int)
t['weekday'] = t[COL['date']].dt.day_name()
if (t[COL['date']].dt.hour > 0).any():
    t['hour'] = t[COL['date']].dt.hour
t = t.rename(columns={COL['date']: 'crash_date'})
COL['date'] = 'crash_date'

• Dropped 0 rows with no valid crash date (blank rows / unparseable dates).
• Dropped 0 rows with a crash date in the future.


### 4.2 Borough and ZIP standardisation
Team rule: borough is title case — `Manhattan, Brooklyn, Queens, Bronx, Staten Island` — and `zip_code` is always a **5-character string**.
We keep the first 5 digits, and set ZIPs outside the NYC range (10001–11697) to missing instead of guessing.
(These are the same rules as Member 3's `standardize_borough` / `standardize_zip`; swap in `helpers.py` once it exists.)

In [ ]:
BOROUGH_MAP = {'MANHATTAN': 'Manhattan', 'NEW YORK': 'Manhattan', 'MN': 'Manhattan',
               'BROOKLYN': 'Brooklyn', 'KINGS': 'Brooklyn', 'BK': 'Brooklyn',
               'QUEENS': 'Queens', 'QN': 'Queens',
               'BRONX': 'Bronx', 'THE BRONX': 'Bronx', 'BX': 'Bronx',
               'STATEN ISLAND': 'Staten Island', 'RICHMOND': 'Staten Island', 'SI': 'Staten Island'}

def standardize_borough(series):
    """Upper-case + strip, map spelling variants to the agreed title-case names; unknown values -> NaN."""
    cleaned = series.astype('string').str.strip().str.upper()
    return cleaned.map(BOROUGH_MAP)

def standardize_zip(series):
    """Keep the first 5 digits as a string (keeps leading zeros); outside the NYC range -> NaN."""
    digits = series.astype('string').str.extract(r'(\d{5})')[0]
    in_range = pd.to_numeric(digits, errors='coerce').between(10001, 11697)
    return digits.where(in_range)

raw_boro = t[COL['borough']].astype('string').str.strip().str.upper()
t['borough'] = standardize_borough(t[COL['borough']])
unmapped = raw_boro[t['borough'].isna() & raw_boro.notna() & (raw_boro != '')].value_counts()
log(f"Borough: {t['borough'].isna().sum():,} rows missing/unrecognised after mapping (labels seen but unmapped: {dict(unmapped.head(5))}).")

zip_before = t[COL['zip']].notna().sum()
t['zip_code'] = standardize_zip(t[COL['zip']])
log(f"ZIP: {zip_before - t['zip_code'].notna().sum():,} ZIPs were malformed or outside 10001-11697 and set to missing.")
t = t.drop(columns=[c for c in {COL['borough'], COL['zip']} if c not in ('borough', 'zip_code')])
print(t['borough'].value_counts(dropna=False))

• Borough: 55 rows missing/unrecognised after mapping (labels seen but unmapped: {}).
• ZIP: 44 ZIPs were malformed or outside 10001-11697 and set to missing.
borough
Brooklyn         3954
Queens           3282
Manhattan        2733
Bronx            2617
Staten Island     669
NaN                55
Name: count, dtype: int64


### 4.3 Coordinates: invalid points
Points at (0, 0) or outside the NYC bounding box (lat 40.49–40.92, lon −74.27 to −73.68) are GPS/entry errors. We set the coordinates to missing
(we do not delete the crash yet, because borough or ZIP may still be valid).

In [ ]:
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 40.49, 40.92, -74.27, -73.68

if COL['lat'] and COL['lon']:
    t = t.rename(columns={COL['lat']: 'latitude', COL['lon']: 'longitude'})
    for c in ('latitude', 'longitude'):
        t[c] = pd.to_numeric(t[c], errors='coerce')
    zero_pt = (t['latitude'] == 0) | (t['longitude'] == 0)
    outside = ~(t['latitude'].between(LAT_MIN, LAT_MAX) & t['longitude'].between(LON_MIN, LON_MAX))
    bad = (zero_pt | outside) & (t['latitude'].notna() | t['longitude'].notna())
    t.loc[bad, ['latitude', 'longitude']] = np.nan
    log(f"Coordinates: {bad.sum():,} points at (0,0) or outside the NYC bounding box set to missing.")
    t.loc[t['latitude'].isna() ^ t['longitude'].isna(), ['latitude', 'longitude']] = np.nan   # keep pairs consistent

• Coordinates: 0 points at (0,0) or outside the NYC bounding box set to missing.


### 4.4 Missing location: impute where defensible, drop where not
A crash needs **at least a borough** to be usable (borough × year is our fallback integration level), and ZIP to be used at ZIP × year.
Decisions:
1. **No ZIP, but valid coordinates → assign the nearest ZIP centroid**, only if the centroid is within ~1.5 km. Centroids come from rows that have both ZIP and coordinates. These rows are flagged `zip_imputed = True`.
2. **Borough missing but ZIP known → take the most common borough of that ZIP.**
3. **Still no borough, no ZIP and no coordinates → drop** (cannot be integrated at any level).
4. Rows with a borough but no ZIP are **kept** (flag `has_zip = False`) so borough × year totals stay complete; they simply don't enter ZIP-level metrics.

*Alternative considered:* dropping every row without a ZIP — rejected because it would bias the data towards areas that report ZIPs better (Member 1 / the missing-ZIP-by-borough table shows this is uneven).

In [ ]:
from scipy.spatial import cKDTree

t['zip_imputed'] = False
if 'latitude' in t.columns:
    has_both = t['zip_code'].notna() & t['latitude'].notna()
    centroids = t[has_both].groupby('zip_code')[['latitude', 'longitude']].mean()
    need = t['zip_code'].isna() & t['latitude'].notna()
    if len(centroids) and need.any():
        # degrees -> km (1 deg lat ~ 111 km, 1 deg lon ~ 84 km at NYC latitude) so the 1.5 km threshold is meaningful
        scale = np.array([111.0, 84.0])
        tree = cKDTree(centroids.to_numpy() * scale)
        dist_km, idx = tree.query(t.loc[need, ['latitude', 'longitude']].to_numpy() * scale)
        ok = dist_km <= 1.5
        target = t.index[need][ok]
        t.loc[target, 'zip_code'] = centroids.index.to_numpy()[idx[ok]]
        t.loc[target, 'zip_imputed'] = True
        log(f"ZIP imputed from nearest ZIP centroid (<=1.5 km) for {ok.sum():,} rows; {(~ok).sum():,} stayed missing.")

# borough from ZIP (most common borough of that ZIP)
zip_to_borough = (t.dropna(subset=['zip_code', 'borough']).groupby('zip_code')['borough']
                    .agg(lambda s: s.mode().iat[0]))
fill = t['borough'].isna() & t['zip_code'].notna()
t.loc[fill, 'borough'] = t.loc[fill, 'zip_code'].map(zip_to_borough)
log(f"Borough filled from ZIP for {t.loc[fill, 'borough'].notna().sum():,} rows.")

# drop rows that cannot be integrated at any level
no_location = t['borough'].isna() & t['zip_code'].isna()
if 'latitude' in t.columns:
    no_location &= t['latitude'].isna()
log(f"Dropped {no_location.sum():,} rows with no borough, ZIP or coordinates.")
t = t[~no_location].copy()
t['has_zip'] = t['zip_code'].notna()
print('Missing ZIP by borough (share of that borough\'s crashes):')
print(t.assign(no_zip=~t['has_zip']).groupby('borough', dropna=False)['no_zip'].mean().round(3))

• ZIP imputed from nearest ZIP centroid (<=1.5 km) for 3,779 rows; 318 stayed missing.
• Borough filled from ZIP for 38 rows.
• Dropped 9 rows with no borough, ZIP or coordinates.
Missing ZIP by borough (share of that borough's crashes):
borough
Bronx            0.105
Brooklyn         0.070
Manhattan        0.109
Queens           0.121
Staten Island    0.348
NaN              1.000
Name: no_zip, dtype: float64


### 4.5 Outliers: domain rules first, IQR second
* **Domain rules:** negative counts are impossible → missing. A single crash with more than 50 people injured/killed (or vehicles) is almost certainly a data-entry error → missing (and logged).
* **IQR (Q3 + 1.5·IQR):** crash count columns are zero-inflated (most crashes have 0 injuries), so the IQR is 0 and the IQR rule would flag *every* non-zero value. We therefore **only flag** with IQR (`flag_<col>_outlier`) and **keep** those rows: severe crashes are real events and are the interesting part of the story. Only the domain-rule errors are removed.

In [ ]:
MAX_PLAUSIBLE = 50
count_cols = [c for c in t.select_dtypes('number').columns
              if re.search(r'injur|kill|death|fatal|pedestrian|cyclist|motorist|vehicle|person|number', c)
              and c not in ('year', 'month', 'hour')]
print('Count columns checked:', count_cols)

outlier_report = []
for c in count_cols:
    neg = t[c] < 0
    huge = t[c] > MAX_PLAUSIBLE
    t.loc[neg | huge, c] = np.nan
    q1, q3 = t[c].quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)
    t[f'flag_{c}_outlier'] = t[c] > upper
    outlier_report.append({'column': c, 'negative_or_>50 set to NaN': int((neg | huge).sum()),
                           'IQR upper bound': round(float(upper), 2), 'flagged by IQR (kept)': int(t[f'flag_{c}_outlier'].sum())})
    log(f"{c}: {int((neg | huge).sum())} impossible values set to missing; {int(t[f'flag_{c}_outlier'].sum())} flagged as IQR outliers and kept.")
pd.DataFrame(outlier_report)

Count columns checked: ['total_non_severe_injuries', 'total_severe_injuries', 'total_fatalities', 'similar_fatalities_severe', 'similar_fatalities_all']
• total_non_severe_injuries: 0 impossible values set to missing; 2171 flagged as IQR outliers and kept.
• total_severe_injuries: 0 impossible values set to missing; 821 flagged as IQR outliers and kept.
• total_fatalities: 0 impossible values set to missing; 1041 flagged as IQR outliers and kept.
• similar_fatalities_severe: 0 impossible values set to missing; 816 flagged as IQR outliers and kept.
• similar_fatalities_all: 11 impossible values set to missing; 2099 flagged as IQR outliers and kept.


,column,negative_or_>50 set to NaN,IQR upper bound,flagged by IQR (kept)
0,total_non_severe_injuries,0,0.0,2171
1,total_severe_injuries,0,1.0,821
2,total_fatalities,0,0.0,1041
3,similar_fatalities_severe,0,0.0,816
4,similar_fatalities_all,11,2.5,2099


### 4.6 Inconsistent text and duplicates
Text columns are stripped, collapsed to single spaces, and empty strings become missing; category labels (contributing factor, vehicle type …) are
upper-cased so `Unspecified` and `UNSPECIFIED` count as one label. Duplicates: first exact duplicates, then duplicates of the **unique record id**
(a collision must appear once).

In [ ]:
text_cols = [c for c in t.select_dtypes(['object', 'string']).columns if c not in ('borough', 'zip_code', 'weekday')]
for c in text_cols:
    s = t[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    t[c] = s.replace({'': pd.NA, 'NAN': pd.NA, 'nan': pd.NA})
    if t[c].nunique() <= 100:
        t[c] = t[c].str.upper()

n0 = len(t)
t = t.drop_duplicates()
log(f"Removed {n0 - len(t):,} exact duplicate rows.")
if COL['id'] and COL['id'] in t.columns:
    n0 = len(t)
    t = t.drop_duplicates(subset=[COL['id']], keep='first')
    log(f"Removed {n0 - len(t):,} rows with a repeated unique id ({COL['id']}).")
else:
    log('No unique id column found: only exact duplicates removed.')
t = t.reset_index(drop=True)

• Removed 0 exact duplicate rows.
• No unique id column found: only exact duplicates removed.


## 5. Validation: before vs after
Row counts and null percentages (shared with Member 6 for the validation plots) and a before/after look at the spatial data.

In [ ]:
rows_after = len(t)
log(f"FINAL: {rows_raw:,} raw rows -> {rows_after:,} clean rows ({rows_raw - rows_after:,} removed, {(rows_raw - rows_after)/rows_raw:.1%}).")
null_after = (t.isna().mean() * 100).round(2)
before_renamed = null_before.rename(index={COL['borough']: 'borough', COL['zip']: 'zip_code', 'crash_date': 'crash_date'})
summary = pd.DataFrame({'null_pct_before': before_renamed, 'null_pct_after': null_after}).dropna(how='all')
display(summary[(summary > 0).any(axis=1)])

fig = go.Figure(go.Bar(x=['Raw sample', 'After cleaning'], y=[rows_raw, rows_after], text=[f'{rows_raw:,}', f'{rows_after:,}'], textposition='outside'))
fig.update_layout(title='Transport row counts before vs after cleaning', yaxis_title='rows'); fig.show()

if 'latitude' in t.columns:
    pts = t[['latitude', 'longitude', 'borough']].dropna()
    s = pts.sample(min(20000, len(pts)), random_state=42)
    px.scatter(s, x='longitude', y='latitude', color='borough', opacity=0.4, title='Clean crash coordinates (all inside the NYC box)').show()

• FINAL: 13,310 raw rows -> 13,301 clean rows (9 removed, 0.1%).


,null_pct_before,null_pct_after
address,80.64,80.63
bbl,82.31,82.30
bin,82.31,82.30
borough,0.41,0.06
census_tract_2020,74.27,74.26
community_board,74.25,74.24
council_district,74.25,74.24
cross_street,39.05,39.06
highway_reference_marker,92.10,92.11
latitude,18.35,18.29


## 6. Save `transport_clean.csv` and the notes file

In [ ]:
t['zip_code'] = t['zip_code'].astype('string')          # 5-character string, never a float
t['year'] = t['year'].astype(int)
t.to_csv(CLEAN_PATH, index=False)

with open(NOTES_PATH, 'w') as f:
    f.write('# Transport cleaning notes (Member 2)\n\n')
    f.write(f'- Dataset: {TRANSPORT_ID} (SIRTA crashes), filter: `{where_clause}`\n')
    f.write(f'- Rows before: {rows_raw:,} | rows after: {rows_after:,}\n')
    f.write(f'- Years covered: {t.year.min()}-{t.year.max()} (latest year may be partial)\n\n## Every filter, drop and assumption\n')
    f.writelines(f'- {m}\n' for m in NOTES)
print('Saved', t.shape, 'to', CLEAN_PATH, 'and', NOTES_PATH)
t.head()

Saved (13301, 33) to data/clean/transport_clean.csv and data/clean/transport_notes.md


,open_data_crash_id,crash_date,crash_time,crash_police_precinct,total_non_severe_injuries,total_severe_injuries,total_fatalities,latitude,longitude,on_street,...,weekday,borough,zip_code,zip_imputed,has_zip,flag_total_non_severe_injuries_outlier,flag_total_severe_injuries_outlier,flag_total_fatalities_outlier,flag_similar_fatalities_severe_outlier,flag_similar_fatalities_all_outlier
0,4492364,2022-01-01,01:28,46,1.0,1.0,0.0,40.858707,-73.90370,JEROME AVENUE,...,Saturday,Bronx,10468,True,True,True,False,False,False,False
1,4491159,2022-01-01,04:33,46,0.0,1.0,0.0,NaN,NaN,JEROME AVENUE,...,Saturday,Bronx,<NA>,False,False,False,False,False,False,False
2,4491487,2022-01-01,05:55,47,0.0,1.0,0.0,40.871550,-73.86428,<NA>,...,Saturday,Bronx,10467,True,True,False,False,False,False,False
3,4491223,2022-01-01,10:42,43,0.0,1.0,0.0,40.823673,-73.87233,BRONX RIVER PARKWAY,...,Saturday,Bronx,10472,True,True,False,False,False,False,False
4,4491418,2022-01-01,20:55,49,0.0,1.0,0.0,40.863945,-73.86452,BOSTON ROAD,...,Saturday,Bronx,10467,True,True,False,False,False,False,False


## 7. Prototype charts 03 and 04 (on the cleaned file)
The repo's chart interface is `TITLE`, `QUESTION`, `build_figure(df, filters)` (see `README_PHASE1.md`). These prototypes use the cleaned crash table;
when Member 3 delivers `integrated_zip_year.csv` we only change the column names (via `config.py`) and aggregate from the integrated table.

In [ ]:
def make_chart_03_prototype(df):
    """Line: crashes per month by borough."""
    if df.empty:
        return go.Figure().add_annotation(text='No data for these filters', showarrow=False)
    m = df.groupby(['year', 'month', 'borough']).size().reset_index(name='crashes')
    m['period'] = pd.to_datetime(dict(year=m.year, month=m.month, day=1))
    return px.line(m, x='period', y='crashes', color='borough', title='Monthly crashes by borough',
                   labels={'period': 'Month', 'crashes': 'Crashes'})

def make_chart_04_prototype(df):
    """Heatmap: crashes by borough x year."""
    if df.empty:
        return go.Figure().add_annotation(text='No data for these filters', showarrow=False)
    pivot = df.pivot_table(index='borough', columns='year', values='crash_date', aggfunc='count', fill_value=0)
    return px.imshow(pivot, text_auto=True, aspect='auto', color_continuous_scale='YlOrRd',
                     title='Crashes by borough and year', labels={'x': 'Year', 'y': 'Borough', 'color': 'Crashes'})

make_chart_03_prototype(t).show()
make_chart_04_prototype(t).show()

## Research questions (Member 2) — draft, finalise after the integrated table exists
**Q5/Chart 03 (line):** How did serious crashes per ZIP change year to year in each borough, and which boroughs saw the biggest rise? *Why it matters:* shows where street safety is getting worse. *Limit:* counts are not rates (no traffic volume); 2026 is a partial year.
**Q6/Chart 04 (heatmap):** In which borough × year combinations are injuries per crash highest, and does that pattern line up with years of heavy housing completions? *Why it matters:* guides where safety programmes might follow new development. *Limit:* association only, not causation; ZIP areas differ in size and population.

## Prompts used (for notebook section 8)
**Prompt 1 (task understanding):** I shared the project brief, dataset catalog, team plan, our Colab file and the GitHub repo, and asked what my role as Member 2 (transportation data owner) involves and what I should do first.
*Claude's answer:* Explained my role and generated a notebook that downloads the 2022-onward SIRTA crash data, runs EDA, cleans the data and saves `transport_clean.csv` and `transport_notes.md`.


**Prompt 2 (dataset choice):** I asked whether the transport dataset had to depend on Member 1's housing dataset, or whether I could choose freely.
*Claude's answer:* The choice depends on Member 1's data. The crash dataset is the only transport option with a ZIP code, so it joins with the housing data at ZIP x year.

**Prompt 3 (workflow):** I asked how to run the notebook, where the cleaned file belongs, and how to commit it to the team GitHub repository.
*Claude's answer:* Run the notebook in Colab, put the cleaned files in `data/clean/`, and commit them on my own branch without the large raw file.


**Prompt 4 (missing values):** I asked whether missing ZIP codes in `transport_clean.csv` are acceptable.
*Claude's answer:* Yes. Rows with a borough but no ZIP are kept on purpose and flagged with `has_zip`, so borough totals stay complete.
